# Advanced Resources

## Limitations of `Resource`

Up to this point the models we have looked have have used a basic `simpy.Resource`.  This has been very useful to model queues, but one potential downside to `Resource` is that it *does not allow you to model individual resource attributes or any type of complex behaviour*. For many models in healthcare, this is sufficient, but there may be instances where you need to track and control individual resources. For example, ambulances, or different types of staff. In this notebook we will explore how to add more complex behaviour using the `Store` and `FilterStore` objects provided by `simpy`.

The good news is that both `Store` and `FilterStore` are easy to use.  Usually this is within the context of a complex simulation, but we will keep our models simple here and focus on how to use them.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import itertools
import simpy
import math

In [2]:
# to reduce code these classes can be found in distribution.py
# you can also pip `install sim-tools`
from distributions import (
    Exponential, 
    DiscreteEmpirical
)

from sim_utility import set_trace, trace, spawn_seeds

## 2. PART 1: Using a `simpy.Store`

A `Store` can be thought of as a "box" that we can `put` or `get` our own custom resource objects from in a First In First Out (FIFO) basis.  **The objects do not need to be of the same type.**

We create an instance of a `Store` as follows (`env` is the `simpy.Environment`):

```python
store = simpy.Store(env, capacity=2)
```
We have to `put` something in the store. For example, let's add some ambulances.

```python
ambulances = [
    Ambulance(ambulance_id=1, vehicle_type="rrv"), 
    Ambulance(ambulance_id=2, vehicle_type="type_1")
]

# loop through list of ambulances and put them in the store object
for amb in ambulances:
    store.put(amb)

```

To get an an ambulance we used the `get()` method along with the `yield` keyword.

```python
ambulance = yield store.get()
```

Like normal `Resource` objects the use of `yield` means that we can simulate queues when no `Ambulance` objects are left in the store.
 
### 2.1 Parameters

In [4]:
NUM_AMBULANCES = 10
FLEET = {
    "rrv": 4,
    "type_1": 6,
}

RRV_SERVICE_TIME   = 50.0   # minutes
TYPE1_SERVICE_TIME = 65.0   
TRAFFIC_INTENSITY  = 0.95   # ρ = λ / (c · μ)
RUN_LENGTH         = 1_000  # minutes
RANDOM_SEED        = 42

# exponential IAT and service time so I have used traffic intensity to control IAT
# ρ = λ / (c · μ)  →  mean_interarrival = mean_service / (ρ · c)
MEAN_INTERARRIVAL = 6 # → 60 / (0.8 × 10) = 7.5 minutes between calls

### 2.2 Entity classes

In [5]:
class Ambulance:
    """
    An ambulance resource

    Parameters:
    ----------
    ambulance_id: int
        Unique id of the ambulance
    """
    def __init__(self, ambulance_id: int, vehicle_type: str):
        self.ambulance_id = ambulance_id
        self.vehicle_type = vehicle_type
        self.total_jobs   = 0
        # cumulative busy time
        self.total_busy   = 0.0          
        
    def __repr__(self):
        """
        A text representation of the ambulance to help debugging
        """
        return f"Ambulance(id={self.ambulance_id})"

In [6]:
class Patient:
    """
    A class to hold patient attributes

    Parameters
    ----------
    patient_id: int
        Unique patient id

    arrival_time: float
        Time of arrival to the simulation    
    """
    def __init__(self, patient_id: int, arrival_time: float):
        self.patient_id   = patient_id
        self.arrival_time = arrival_time

    def __repr__(self):
        """
        A text representation of the Patient for debug
        """
        return f"Patient(id={self.patient_id})"

## 2.3 Ambulance dispatch and service process

In [7]:
def dispatch_ambulance(
    env: simpy.Environment,
    store: simpy.Store,
    patient: Patient,
    dists: dict,
    log: dict
) -> None:
    """Simulate ambulance dispatch and service: 
    
    1. queues a patient up for the next free Ambulance (FIFO), 
    2. simulates service (as a single distribution), 
    3. returns the Ambulance to the store.

    Parameters:
    ----------
    env: simpy.Environment
        The simpy environment for the simulation
    store: simpy.Store
        A store of Ambulance objects
    dists: dict
        Contains the "service" distribution
    log: dict
        Audit dictionary
    """

    # Wait for an available Ambulance
    # note we `get()` an Ambulance from the store
    ambulance: Ambulance = yield store.get()

    wait_time = env.now - patient.arrival_time
    log["wait_times"].append(wait_time)

    # Service varies by vehicle type (travel + on-scene + return)
    service_dist = dists[ambulance.vehicle_type]
    service_time = service_dist.sample()

    # debug
    trace(
        f"{env.now:.1f}  {patient} → {ambulance} "
        f"[{ambulance.vehicle_type}] "
        f"(waited {wait_time:.1f} min, service {service_time:.1f} min)"
    )

    yield env.timeout(service_time)

    # Update ambulance stats and return (put) to store
    ambulance.total_jobs += 1
    ambulance.total_busy += service_time
    store.put(ambulance)

    log["service_times"].append(service_time)
    log["assignments"].append(
        (patient.patient_id, ambulance.ambulance_id, ambulance.vehicle_type)
    )

### 2.4 Patient arrival generator

In [8]:
def patient_arrivals_generator(
    env: simpy.Environment,
    store: simpy.Store,
    distributions: dict,
    results: dict
) -> None:
    """
    Arrival process for patients to the ambulance sim.

    Parameters:
    ------
    env: simpy.Environment
        The simpy environment for the simulation

    store: simpy.Store
        A store of Ambulance objects

    distributions: dict
        Contains "arrival" and "service" distributions

    results: dict
        Results dictionary
    """
    for patient_id in itertools.count(start=1):

        # time until next patient arrival
        inter_arrival_time = distributions["arrival"].sample()
        yield env.timeout(inter_arrival_time)

        results["n_arrivals"] += 1
        patient = Patient(patient_id, env.now)

        # debug info
        trace(f"{env.now:.1f}: Arrival. {patient}")

        # create ambulance dispatch + service process
        env.process(dispatch_ambulance(env, store, patient, distributions, results))

### 2.5 Single run function

In [9]:
def single_run(
    mean_iat: float = MEAN_INTERARRIVAL,
    mean_rrv_service: float = RRV_SERVICE_TIME,
    mean_type_1_service: float = TYPE1_SERVICE_TIME,
    n_ambulances: int = NUM_AMBULANCES,
    fleet: list = FLEET,
    run_length: float = RUN_LENGTH, 
    random_seed: int = 1
):
    """
    Set up and perform a single replication of the MMS model
    """

    # generate 3 rng seeds
    seeds = spawn_seeds(n_streams=3, main_seed=random_seed)
    
    # 1. distribution objects
    dists = {
        "arrival": Exponential(mean_iat, random_seed=seeds[0]),
        "rrv": Exponential(mean_rrv_service, random_seed=seeds[1]),
        "type_1": Exponential(mean_type_1_service, random_seed=seeds[2]),
    }

    # 2. simpy environment 
    env = simpy.Environment()

    # 3. Initialise Store
    # 3.1 Create empty Store with sufficient slots
    store = simpy.Store(env, capacity=n_ambulances)

    # 3.2 Create Ambulance objects
    ambulances = []
    ambulance_id = 0
    
    for vehicle_type, count in fleet.items():
        for _ in range(count):
            ambulance_id += 1
            ambulances.append(Ambulance(ambulance_id, vehicle_type))
    
    # 3.3 `put` Ambulance objects into the store
    for amb in ambulances:
        store.put(amb)

    # 4. results dictionary
    log = {"n_arrivals": 0, "wait_times": [], "service_times": [], "assignments": []}

    env.process(patient_arrivals_generator(env, store, dists, log))
    env.run(until=run_length)

    return ambulances, log

In [11]:
set_trace(True)
# no warm-up.
ambulances, log = single_run(random_seed=42)

waits = np.array(log["wait_times"])
services = np.array(log["service_times"])
n = len(waits)

print("\n" + "═" * 55)
print(f"  Mean inter-arrival   : {MEAN_INTERARRIVAL:.2f} min")
print("─" * 55)
print(f"  Patients arrived     : {log['n_arrivals']}")
print(f"  Patients served      : {n}")
print(f"  Mean wait time       : {waits.mean():.2f} min")
print(f"  P(wait > 0)          : {(waits > 0).mean():.2%}")
print(f"  95th pct wait        : {np.percentile(waits, 95):.2f} min")
print("─" * 55)
print(f"  {'Ambulance':<15} {'Jobs':>6} {'Utilisation':>12}")
print("─" * 55)

# calculate utilisation of each individual ambulance
for amb in ambulances:
    util = amb.total_busy / RUN_LENGTH
    print(f"  Ambulance {amb.ambulance_id:<5}     {amb.total_jobs:>6}       {util:>8.2%}")
print("═" * 55)

Simulation tracing set to: True
5.8: Arrival. Patient(id=1)
5.8  Patient(id=1) → Ambulance(id=1) [rrv] (waited 0.0 min, service 9.7 min)
16.8: Arrival. Patient(id=2)
16.8  Patient(id=2) → Ambulance(id=2) [rrv] (waited 0.0 min, service 4.4 min)
19.5: Arrival. Patient(id=3)
19.5  Patient(id=3) → Ambulance(id=3) [rrv] (waited 0.0 min, service 24.1 min)
21.2: Arrival. Patient(id=4)
21.2  Patient(id=4) → Ambulance(id=4) [rrv] (waited 0.0 min, service 7.6 min)
40.8: Arrival. Patient(id=5)
40.8  Patient(id=5) → Ambulance(id=5) [type_1] (waited 0.0 min, service 12.3 min)
41.8: Arrival. Patient(id=6)
41.8  Patient(id=6) → Ambulance(id=6) [type_1] (waited 0.0 min, service 22.9 min)
42.1: Arrival. Patient(id=7)
42.1  Patient(id=7) → Ambulance(id=7) [type_1] (waited 0.0 min, service 8.4 min)
43.0: Arrival. Patient(id=8)
43.0  Patient(id=8) → Ambulance(id=8) [type_1] (waited 0.0 min, service 91.4 min)
69.8: Arrival. Patient(id=9)
69.8  Patient(id=9) → Ambulance(id=9) [type_1] (waited 0.0 min, servi

## 3. PART 2: Using a `FilterStore`

In this example we will modify the ambulance dispatch simulation so that patients and ambulances are located within one of five **nodes** that are specificed by coordinates.

| ID | Label | Coordinates |
|---|---|---|
| 0 | South-West | `(0, 0)` |
| 1 | South-East | `(4, 0)` |
| 2 | North-West | `(0, 4)` |
| 3 | North-East | `(4, 4)` |
| 4 | Centre | `(2, 2)` |
| 5 | Hospital | `(2, 0)` |



Patients in need will be assigned the closest available ambulance. If not ambulances are available they are assigned the next available ambulance when it has returned to its home node.

### 3.1 Geographic information

In [ ]:
# All locations share a single distance matrix: nodes 0-4 plus hospital (5)
HOSPITAL_ID = 5

LOCATIONS = {
    0: (0.0, 0.0),   # South-West
    1: (4.0, 0.0),   # South-East
    2: (0.0, 4.0),   # North-West
    3: (4.0, 4.0),   # North-East
    4: (2.0, 2.0),   # Centre
    HOSPITAL_ID: (2.0, 0.0),  # Hospital (south-centre)
}

# nodes patients can appear in
PATIENT_NODES = [0, 1, 2, 3, 4]   
NODE_PROBS = [0.15, 0.25, 0.20, 0.20, 0.20]

# Pre-compute ALL pairwise distances before simulation
DISTANCES = {
    (i, j): math.dist(LOCATIONS[i], LOCATIONS[j])
    for i in LOCATIONS
    for j in LOCATIONS
}

### 3.2 Parameters

In [ ]:
NUM_AMBULANCES    = 10
TRAVEL_SPEED      = 0.25  # distance units per minute
MEAN_SCENE_TIME   = 20.0  
MEAN_INTERARRIVAL = 6    
RUN_LENGTH        = 1_000
RANDOM_SEED       = 42

# Ambulance positioning parameter
# 2 ambulances stationed at each of the 5 nodes
AMBULANCE_HOME_NODES = [
    node
    for node in PATIENT_NODES
    for _ in range(NUM_AMBULANCES // len(PATIENT_NODES))
]

### 3.3 Entity Classes

The `Ambulance` and `Patient` classes have been slightly modified to include a `node` attribute

In [ ]:
class Ambulance:
    def __init__(self, ambulance_id: int, home_node: int):
        self.ambulance_id = ambulance_id
        # modification = a home node or 'base' for the ambulance
        self.home_node    = home_node
        self.total_jobs   = 0
        self.total_busy   = 0.0

    def __repr__(self):
        return f"Ambulance(id={self.ambulance_id}, node={self.home_node})"

In [ ]:
class Patient:
    def __init__(self, patient_id: int, arrival_time: float, node: int):
        self.patient_id  = patient_id
        self.arrival_time = arrival_time
        self.node = node

    def __repr__(self):
        return f"Patient(id={self.patient_id}, node={self.node})"

### 3.4 Travel functions

In [ ]:
def travel_time(loc_a: int, loc_b: int) -> float:
    """Minutes to travel between any two location IDs."""
    return DISTANCES[loc_a, loc_b] / TRAVEL_SPEED

In [ ]:
def closest_ambulance_id(
    available: list[Ambulance],
    patient_node: int
) -> int | None:
    
    if not available:
        return None
        
    return min(available, key=lambda a: DISTANCES[a.home_node, patient_node]).ambulance_id

### 3.5 Modified simpy processes

The biggest update is to our ambulance dispatch function that will now use a `FilterStore` to take account of `Patient` and `Ambulance` node location.

In [ ]:
def dispatch_ambulance(env, store, patient, dists, log):
    """Modified ambulance dispatch process
    """
   
    # find the closest ambulance in the store
    # note we pass store.items which is a list of Ambulance objects
    # it may be empty!
    best_id = closest_ambulance_id(store.items, patient.node)

    if best_id is not None:
        # if an ambulance is available get the closest ambulance from the store
        ambulance = yield store.get(lambda a: a.ambulance_id == best_id)
    else:
        # other wise just wait for the next available ambulance. FIFO
        ambulance = yield store.get(lambda a: True)

    wait_time = env.now - patient.arrival_time

    # Leg 1: travel from home node to patient 
    t_to_patient = travel_time(ambulance.home_node, patient.node)

    # Leg 2: on scene
    scene_time = dists["on_scene"].sample()

    # Leg 3: transport patient to hospital
    t_to_hospital = travel_time(patient.node, HOSPITAL_ID)

    # Leg 4: return to home base 
    t_to_base = travel_time(HOSPITAL_ID, ambulance.home_node)

    # total turnaround time
    total_service = t_to_patient + scene_time + t_to_hospital + t_to_base
    yield env.timeout(total_service)
    
    ambulance.total_jobs += 1
    ambulance.total_busy += total_service
    store.put(ambulance)

    log["wait_times"].append(wait_time)

    # create row in the assignments table
    log["assignments"].append(
        dict(patient_id=patient.patient_id,
             patient_node=patient.node,
             ambulance_id=ambulance.ambulance_id,
             ambulance_node=ambulance.home_node,
             dispatch_dist=DISTANCES[ambulance.home_node, patient.node],
             t_to_patient=t_to_patient,
             scene_time=scene_time,
             t_to_hospital=t_to_hospital,
             t_to_base=t_to_base,
             total_service=total_service,
             immediate_dispatch=best_id is not None,
             wait=wait_time)
    )

In [ ]:
def patient_arrivals_generator(
    env: simpy.Environment,
    store: simpy.Store,
    dists: dict,
    log: dict
) -> None:       
    """Modified Arrival process for patients to the ambulance sim
    Now includes arrival node"""
    for patient_id in itertools.count(start=1):

        # time until next patient arrival
        inter_arrival_time = dists["arrival"].sample()
        yield env.timeout(inter_arrival_time)

        log["n_arrivals"] += 1
        node  = dists["arrival_node"].sample()
        patient = Patient(patient_id, env.now, node)

        # debug info
        trace(f"{env.now:.1f}: Arrival. {patient}")

        # create ambulance dispatch + service process
        env.process(dispatch_ambulance(env, store, patient, dists, log))

In [ ]:
def single_run(
    mean_iat: float = MEAN_INTERARRIVAL,
    mean_on_scene: float = MEAN_SCENE_TIME,
    n_ambulances: int = NUM_AMBULANCES,
    run_length: float = RUN_LENGTH, 
    random_seed: int = 42
):
    """
    Set up and perform a single replication of the MMS model
    """

    # generate 3 rng seeds
    seeds = spawn_seeds(n_streams=3, main_seed=random_seed)
    
    # 1. distribution objects
    # We have included a new arrival_node an on_scene dists
    dists = {
        "arrival": Exponential(mean_iat, random_seed=seeds[0]),
        "arrival_node": DiscreteEmpirical(
            values=[0, 1, 2, 3, 4], 
            freq=[p * 100 for p in NODE_PROBS],
            random_seed=seeds[1]),
        "on_scene": Exponential(mean_on_scene, random_seed=seeds[2]),
    }

    # 2. simpy environment 
    env = simpy.Environment()

    # 3. Initialise Store
    # 3.1 Create empty Store with sufficient slots
    store = simpy.FilterStore(env, capacity=n_ambulances)

    # 3.2 Create Ambulance objects now with home nodes
    ambulances = [Ambulance(i + 1, home) for i, home in enumerate(AMBULANCE_HOME_NODES)]

    # 3.3 `put` Ambulance objects into the filter store
    for amb in ambulances:
        store.put(amb)

    # 4. results dictionary
    log = {"n_arrivals": 0, "wait_times": [], "service_times": [], "assignments": []}

    env.process(patient_arrivals_generator(env, store, dists, log))
    env.run(until=run_length)

    return ambulances, log

In [ ]:
def single_run_summary(assignments, ambulances):
    waits = assignments["wait"].to_numpy()
    
    # immediate versus delayed dispatches of Ambulance
    immediate  = assignments[assignments["immediate_dispatch"]]
    delayed = assignments[~assignments["immediate_dispatch"]]
    
    print("\n" + "═" * 65)
    print(f"  Patients served            : {len(assignments)}")
    print(f"  Immediate Dispatch         : {len(immediate)}   ({len(immediate)/len(assignments):.1%})")
    print(f"  Delayed Dispatch           : {len(delayed)}  ({len(delayed)/len(assignments):.1%})")
    print(f"  Overall mean wait (min)    : {waits.mean():.2f}")
    print("─" * 65)
    print(f"  {'':<30} {'Ambulance Dispatch':>20}")
    print(f"  {'Metric':<30} {'Immediate':>10} {'Delayed':>10}")
    print("─" * 65)
    for metric, col in [("Mean wait (min)",         "wait"),
                        ("Mean dispatch dist",       "dispatch_dist"),
                        ("Mean travel to patient",   "t_to_patient"),
                        ("Mean scene time",          "scene_time"),
                        ("Mean travel to hospital",  "t_to_hospital"),
                        ("Mean return to base",      "t_to_base"),
                        ("Mean total service",       "total_service")]:
        c = immediate[col].mean()  if len(immediate)  else float("nan")
        f = delayed[col].mean() if len(delayed) else float("nan")
    
        print(f"  {metric:<30} {c:>10.2f} {f:>10.2f}")
    print("─" * 65)
    print(f"  {'Ambulance':<14} {'Node':>10} {'Jobs':>8} {'Util':>10}")
    print("─" * 65)
    for amb in ambulances:
        print(f"  Ambulance {amb.ambulance_id:<4}      "
              f"{amb.home_node:>5}  {amb.total_jobs:>6}    "
              f"{amb.total_busy / RUN_LENGTH:>8.2%}")
    print("═" * 65)

In [ ]:
# run model
ambulances, log = single_run(random_seed=42)

# get all of the patient to ambulance assignment details
assignments = pd.DataFrame(log["assignments"])
assignments.tail()

In [ ]:
ambulances, log = single_run(random_seed=42)
single_run_summary(assignments, ambulances)